In [5]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Standard Pascal VOC 20 Klassen (wie in offline_test.py definiert)
VOC_CLASSES = [
    "aeroplane", "bicycle", "bird", "boat", "bottle",
    "bus", "car", "cat", "chair", "cow",
    "diningtable", "dog", "horse", "motorbike", "person",
    "pottedplant", "sheep", "sofa", "train", "tvmonitor"
]

def generate_diagnostic_figure(csv_path: str = "voc_behavioral_results.csv", 
                               output_path: str = "yolo26n_diagnostic_dashboard.png"):
    # Daten laden und NaN-Werte in Predicted_Class abfangen
    df = pd.read_csv(csv_path)
    df["Predicted_Class"] = df["Predicted_Class"].fillna("None")

    plt.style.use("seaborn-v0_8-whitegrid")
    fig, axes = plt.subplots(2, 2, figsize=(16, 12), dpi=150)
    fig.suptitle(
        f"YOLO26n Behavioral & Diagnostic Analysis on Pascal VOC (n = {len(df):,} Records)",
        fontsize=15, fontweight="bold", y=0.98
    )

    # =========================================================================
    # Panel A: Joint Phase Space: Confidence vs. IoU across Scales
    # =========================================================================
    ax1 = axes[0, 0]
    palette_outcomes = {
        "TP": "#2ca02c",
        "Missed Target": "#7f7f7f",
        "False Positive": "#ff7f0e",
        "Misclassification": "#d62728"
    }
    markers_scale = {
        "Medium Scale (4-20%)": "o",
        "Large Scale (>20%)": "X",
        "Small Scale (<4%)": "s"
    }

    sns.scatterplot(
        data=df,
        x="Confidence",
        y="IoU",
        hue="Outcome",
        style="Environmental_Condition",
        palette=palette_outcomes,
        markers=markers_scale,
        alpha=0.75,
        s=45,
        ax=ax1
    )
    ax1.axhline(0.50, color="#e41a1c", linestyle="--", alpha=0.7, label="IoU Threshold (0.50)")
    ax1.axvline(0.25, color="#377eb8", linestyle=":", alpha=0.7, label="Conf Threshold (0.25)")
    ax1.set_title("A. Joint Phase Space: Confidence vs. IoU across Scales", fontsize=12, fontweight="bold")
    ax1.set_xlabel("Prediction Confidence Score", fontsize=10)
    ax1.set_ylabel("Intersection over Union (IoU)", fontsize=10)
    ax1.set_xlim(-0.05, 1.0)
    ax1.set_ylim(-0.05, 1.05)
    ax1.legend(loc="lower right", fontsize=8, frameon=True)

    # =========================================================================
    # Panel B: Scale Sensitivity: Outcome Proportions by Relative Box Area
    # =========================================================================
    ax2 = axes[0, 1]
    scale_order = ["Small Scale (<4%)", "Medium Scale (4-20%)", "Large Scale (>20%)"]
    outcome_order = ["TP", "Misclassification", "Missed Target", "False Positive"]
    colors_b = ["#2ca02c", "#d62728", "#7f7f7f", "#ff7f0e"]

    ct_pct = pd.crosstab(
        df["Environmental_Condition"],
        df["Outcome"],
        normalize="index"
    ).loc[scale_order, outcome_order] * 100

    bottom = np.zeros(len(scale_order))
    x_pos = np.arange(len(scale_order))

    for outcome, color in zip(outcome_order, colors_b):
        values = ct_pct[outcome].values
        ax2.bar(x_pos, values, bottom=bottom, color=color, width=0.6, label=outcome, edgecolor="none")
        for i, (val, b) in enumerate(zip(values, bottom)):
            if val >= 1.2:
                ax2.text(
                    i, b + val / 2, f"{val:.1f}%",
                    ha="center", va="center", color="white", fontweight="bold", fontsize=8.5
                )
        bottom += values

    ax2.set_title("B. Scale Sensitivity: Outcome Proportions by Relative Box Area", fontsize=12, fontweight="bold")
    ax2.set_xticks(x_pos)
    ax2.set_xticklabels(scale_order, fontsize=10)
    ax2.set_xlabel("Object Scale Category (% of Image Area)", fontsize=10)
    ax2.set_ylabel("Percentage of Records (%)", fontsize=10)
    ax2.set_ylim(0, 105)
    ax2.legend(title="Outcome", loc="upper right", fontsize=8.5, title_fontsize=9, frameon=True)

    # =========================================================================
    # Panel C: Class Diagnostic Quadrant: Recall vs. Precision (Size = GT Count)
    # =========================================================================
    ax3 = axes[1, 0]
    class_metrics = []
    for cls in VOC_CLASSES:
        gt_sub = df[df["Ground_Truth"] == cls]
        gt_count = len(gt_sub)
        tp = (gt_sub["Outcome"] == "TP").sum()
        mis_gt = (gt_sub["Outcome"] == "Misclassification").sum()
        fp_cls = ((df["Predicted_Class"] == cls) & (df["Outcome"] == "False Positive")).sum()

        recall = (tp / gt_count) * 100 if gt_count > 0 else 0.0
        precision = (tp / (tp + fp_cls + mis_gt)) * 100 if (tp + fp_cls + mis_gt) > 0 else 0.0

        class_metrics.append({
            "Class": cls,
            "GT_Count": gt_count,
            "Recall": recall,
            "Precision": precision,
            "FP_Count": fp_cls
        })

    cdf = pd.DataFrame(class_metrics)

    sc = ax3.scatter(
        cdf["Recall"],
        cdf["Precision"],
        s=cdf["GT_Count"] * 2.2,
        c=cdf["FP_Count"],
        cmap="YlOrRd",
        alpha=0.85,
        edgecolors="black",
        linewidth=1.0
    )

    for _, row in cdf.iterrows():
        ax3.text(
            row["Recall"] + 0.4, row["Precision"] + 0.4, row["Class"],
            fontsize=8, fontweight="bold", color="#222222"
        )

    ax3.axvline(80, color="gray", linestyle="--", alpha=0.6)
    ax3.axhline(70, color="gray", linestyle="--", alpha=0.6)
    ax3.set_title("C. Class Diagnostic Quadrant: Recall vs. Precision (Size = GT Count)", fontsize=12, fontweight="bold")
    ax3.set_xlabel("True Positive Recall Rate (%)", fontsize=10)
    ax3.set_ylabel("Precision (%)", fontsize=10)
    ax3.set_xlim(60, 102)
    ax3.set_ylim(25, 100)

    cbar = plt.colorbar(sc, ax=ax3)
    cbar.set_label("False Positive Count", fontsize=9)

    # =========================================================================
    # Panel D: Deep Error Taxonomy Breakdown (Fine-Grained Failure Modes)
    # =========================================================================
    ax4 = axes[1, 1]

    tp_count = int((df["Outcome"] == "TP").sum())
    fp_small = int(((df["Outcome"] == "False Positive") & (df["Environmental_Condition"] == "Small Scale (<4%)")).sum())
    fp_med_large = int(((df["Outcome"] == "False Positive") & (df["Environmental_Condition"] != "Small Scale (<4%)")).sum())
    missed_loc = int(((df["Outcome"] == "Missed Target") & (df["IoU"] > 0)).sum())
    missed_blind = int(((df["Outcome"] == "Missed Target") & (df["IoU"] == 0)).sum())
    misclass_intra = int(((df["Outcome"] == "Misclassification") & (df["Predicted_Class"].isin(VOC_CLASSES))).sum())
    misclass_oov = int(((df["Outcome"] == "Misclassification") & (~df["Predicted_Class"].isin(VOC_CLASSES))).sum())

    categories = [
        "TP\n(IoU ≥ 0.5, Correct)",
        "FP: Small Scale\n(<4% Area)",
        "FP: Med/Large\n(≥4% Area)",
        "Missed: Loc. Error\n(0 < IoU < 0.5)",
        "Missed: Blind Spot\n(IoU = 0.0)",
        "Misclass: Intra-VOC\n(Semantic Error)",
        "Misclass: COCO OOV\n(Taxonomy Shift)"
    ][::-1]

    counts = [
        tp_count,
        fp_small,
        fp_med_large,
        missed_loc,
        missed_blind,
        misclass_intra,
        misclass_oov
    ][::-1]

    colors_d = [
        "#2ca02c",
        "#ff7f0e",
        "#ffbb78",
        "#1f77b4",
        "#7f7f7f",
        "#d62728",
        "#9467bd"
    ][::-1]

    y_pos = np.arange(len(categories))
    bars_d = ax4.barh(y_pos, counts, color=colors_d, edgecolor="black", height=0.65, linewidth=0.8)

    for bar, count in zip(bars_d, counts):
        ax4.text(
            bar.get_width() + 12, bar.get_y() + bar.get_height() / 2,
            f"{count}", va="center", ha="left", fontsize=9, fontweight="bold"
        )

    ax4.set_yticks(y_pos)
    ax4.set_yticklabels(categories, fontsize=9.5)
    ax4.set_title("D. Deep Error Taxonomy Breakdown (Fine-Grained Failure Modes)", fontsize=12, fontweight="bold")
    ax4.set_xlabel("Number of Instances", fontsize=10)
    ax4.set_xlim(0, 1150)

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig(output_path, dpi=150)
    plt.close(fig)
    print(f"Dashboard erfolgreich gespeichert unter: {output_path}")

if __name__ == "__main__":
    # Projektordner, in dem die CSV und die Skripte liegen
    base_dir = Path(r"C:\Users\Simon Mader\Documents\AI\3. Semester\AI_and_visualization\AI_visualization\Andres-Simon\phase1")
    
    csv_file = base_dir / "voc_behavioral_results.csv"
    output_img = base_dir / "yolo26n_diagnostic_dashboard.png"
    
    if not csv_file.exists():
        raise FileNotFoundError(
            f"Datei nicht gefunden unter: {csv_file}\n"
            "Bitte führe zuerst 'python offline_test.py' in diesem Verzeichnis aus."
        )
        
    generate_diagnostic_figure(
        csv_path=str(csv_file),
        output_path=str(output_img)
    )

Dashboard erfolgreich gespeichert unter: C:\Users\Simon Mader\Documents\AI\3. Semester\AI_and_visualization\AI_visualization\Andres-Simon\phase1\yolo26n_diagnostic_dashboard.png
